# Module 10 project-exercise demo — Contextual cache keys and attention-output error

**Implementation demo, not a completed full-scale project.** Demo 1 models pages and exact token-prefix matches, distinguishing request hits from reused-token fraction. Demo 2 measures attention-output error after teaching quantization and eviction. These CPU mechanisms use synthetic data; they are not vLLM, KIVI, TurboQuant, or H2O implementations.
All inputs are small, deterministic, offline, and CPU-friendly. The goal is to
validate interfaces, metrics, invariants, and report structure before spending
GPU or API budget.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"reproducible smoke tier; seed={SEED}")
DEMO_LANGUAGE = 'en'

plt.rcParams.update({"font.family":"DejaVu Sans", "font.size":12, "axes.titlesize":15, "axes.labelsize":13, "xtick.labelsize":11, "ytick.labelsize":11, "figure.dpi":150})


## Demo 1. Unused tails and matching prefixes

Allocate 16-slot pages for 500 random lengths and compare them with reserving 4096 slots per request. Then issue five 48-token requests for each of 100 users. The first 32 tokens come from one of ten shared templates; the remainder distinguishes user and turn. Cache identity includes the entire preceding prefix, access scope, and model revision. Matching an isolated block is insufficient.

In [ ]:
import hashlib
rng = np.random.default_rng(SEED)
lengths = rng.integers(64, 4097, size=500)
page = 16
allocated = ((lengths + page - 1) // page) * page
reserved = np.full_like(lengths, 4096)
assert np.all(allocated - lengths < page)
print("waste / allocated:", (allocated-lengths).sum()/allocated.sum(),
      "fixed-reservation waste:", (reserved-lengths).sum()/reserved.sum())
assert (2048 + page - 1) // page * page == 2048

# Complete block boundaries. The key includes ALL preceding token IDs.
# No real KV tensors, tokenizer, network, TTL, eviction, or serving engine.
def prefix_key(tokens, end, scope, revision):
    return (scope, revision, tuple(tokens[:end]))

def run_prefix_trace(isolated):
    cache = set(); hit_requests = cached_tokens = input_tokens = 0
    for user in range(100):
        for turn in range(5):
            tokens = [100 + user % 10] + list(range(31)) + [1000+user, 2000+turn] + list(range(14))
            scope = user if isolated else "shared-public-template"
            hit = 0
            for end in range(page, len(tokens)+1, page):
                if prefix_key(tokens,end,scope,"model-r1") in cache:
                    hit = end
                else:
                    break
            for end in range(page,len(tokens)+1,page):
                cache.add(prefix_key(tokens,end,scope,"model-r1"))
            hit_requests += hit > 0
            cached_tokens += hit
            input_tokens += len(tokens)
    return hit_requests/500, cached_tokens/input_tokens

shared = run_prefix_trace(False); isolated = run_prefix_trace(True)
assert np.isclose(shared[0], .98) and np.isclose(isolated[0], .8)
assert np.isclose(shared[1], 490*32/(500*48))
print("request-hit fraction / reused-input fraction:", shared, isolated)
a = list(range(32)); b = a.copy(); b[0] = 999
assert a[16:32] == b[16:32]  # same block content, different preceding context
assert prefix_key(a,32,"scope","r1") != prefix_key(b,32,"scope","r1")
assert prefix_key(a,32,"scope","r1") != prefix_key(a,32,"scope","r2")
print("Context and model-revision identity: PASS")

The shared scope yields a 98% request-hit fraction; per-user isolation yields 80%. Reused-input fractions are only about 65.33% and 53.33%, because each request still has sixteen new tokens. None of these rates alone predicts speedup. In the allocation experiment, exactly 2048 tokens fill every page; the average waste across random lengths belongs to a different distribution.

## Demo 2. Attention error when the query changes

Generate 128 keys and values of width 32 and twelve queries. The teaching quantizer uses symmetric integer codes with one scale per row. Reconstruct the arrays and compare attention outputs. A second experiment makes an earlier query favor position 0 and the next query require position 90. Keeping only 16 positions exposes the limits of decisions based on past attention or token position alone.

In [ ]:
rng = np.random.default_rng(SEED + 10)
n, d = 128, 32
keys = rng.normal(size=(n,d)); values = rng.normal(size=(n,d))
queries = rng.normal(size=(12,d))
def attention(q,k,v):
    logits = q @ k.T / np.sqrt(k.shape[1])
    logits -= logits.max(axis=-1,keepdims=True)
    prob = np.exp(logits); prob /= prob.sum(axis=-1,keepdims=True)
    return prob @ v, prob

def fake_quantize_rows(x,bits):
    limit = 2**(bits-1)-1
    scale = np.max(np.abs(x),axis=-1,keepdims=True)/limit
    scale = np.where(scale == 0, 1.0, scale)
    codes = np.clip(np.rint(x/scale),-limit,limit).astype(np.int8)
    # Float64 dequantization here is a numerical experiment, not a packed kernel.
    return codes.astype(float)*scale

reference,_ = attention(queries,keys,values)
for bits in (8,3):
    kq,vq = fake_quantize_rows(keys,bits),fake_quantize_rows(values,bits)
    out,_ = attention(queries,kq,vq)
    error = np.linalg.norm(out-reference)/np.linalg.norm(reference)
    ideal_codes = 2*n*d*bits/8
    fp32_scales = 2*n*4
    print(f"{bits} bits: relative output error={error:.5f}; ideal packed codes+scales={ideal_codes+fp32_scales:.0f} B")
    if bits == 8: assert error < .03
np.testing.assert_array_equal(fake_quantize_rows(np.zeros((2,4)),3),0)

# A controlled query shift: past attention favors token 0; the next query needs token 90.
k = np.zeros((n,d)); k[0,0] = 12; k[90,1] = 12
v = np.zeros((n,2)); v[0,0] = 1; v[90,1] = 1
past = np.zeros((1,d)); past[0,0] = 4
future = np.zeros((1,d)); future[0,1] = 4
_, mass = attention(past,k,v)
budget = 16
ranked = np.argsort(-mass[0],kind="stable")[:budget]
streaming = np.r_[np.arange(4),np.arange(n-(budget-4),n)]
exact,_ = attention(future,k,v)
for name,idx in (("past top-mass",ranked),("sink + recent",streaming)):
    out,_ = attention(future,k[idx],v[idx])
    error = np.linalg.norm(out-exact)
    print(name,"retains future target:",90 in idx,"output error:",error)
    assert 90 not in idx and error > .9
# Keeping the target has a low error on this example, but requires future knowledge.
oracle = np.r_[90,np.arange(budget-1)]
recovered,_ = attention(future,k[oracle],v[oracle])
assert np.linalg.norm(recovered-exact) < .03
print("Query-shift failure and oracle control: PASS")

The byte estimate includes ideal packed codes and float32 scales. It is not the size of the NumPy arrays, whose reconstructed values are float64. After the query shift, both retention rules discard position 90 and incur a large output error. The oracle control keeps that position and recovers the output, but assumes knowledge of the future query. It diagnoses the failure rather than supplying a deployable prediction rule.

## Scaling to the full project

For E4, first vary length distribution and template count. For E5, vary bit width, retention budget, and future target position. An optional engine experiment should pin arrivals, warm-up, revisions, and SLOs, then measure TTFT/ITL, goodput, memory, and answer quality. Retained K/V norm is not a substitute for response quality.

### Minimum report

1. Data, model, and code revisions, plus the random seed.
2. The primary metric, plus quality, cost, latency, and memory where applicable.
3. A baseline, a matched budget, and at least one negative control.
4. Failed runs and the limits of the supported conclusions.